In [2]:
import pandas as pd

# Đọc dữ liệu
df = pd.read_csv('../data/all_reviews.csv')

# Đếm số lần rating của từng user
user_counts = df['username'].value_counts()

# Chọn user có từ 10 rating trở lên
active_users = user_counts[user_counts >= 20].index.tolist()

print(f'Số lượng user thỏa mãn: {len(active_users)}')

Số lượng user thỏa mãn: 183


In [3]:
filtered_df = df[df['username'].isin(active_users)]

print(f'Số lượng dòng dữ liệu sau lọc: {filtered_df.shape[0]}')

Số lượng dòng dữ liệu sau lọc: 8298


In [4]:
# Tạo tập testing: với mỗi user, lấy ngẫu nhiên 5 dòng
testing_df = filtered_df.groupby('username').sample(n=5, random_state=42)

# Tạo tập training bằng cách loại bỏ các dòng trong tập testing khỏi filtered_df
training_df = pd.concat([filtered_df, testing_df]).drop_duplicates(keep=False)

# Kiểm tra số lượng
print(f'Số lượng dòng tập testing: {testing_df.shape[0]}')
print(f'Số lượng dòng tập training: {training_df.shape[0]}')

Số lượng dòng tập testing: 915
Số lượng dòng tập training: 5995


In [5]:
testing_df.to_csv('../data/collabrative/testing_set.csv', index=False)
training_df.to_csv('../data/collabrative/training_set.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Đọc dữ liệu
training_df = pd.read_csv('../data/collabrative/training_set.csv')
testing_df = pd.read_csv('../data/collabrative/testing_set.csv')

# Tạo ma trận user-item từ training set
user_item_matrix = training_df.pivot_table(index='username', columns='id_product', values='rating', fill_value=0)

# Tính độ tương đồng cosine giữa các user
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

# Hàm dự đoán rating
def predict_ratings(user_item_matrix, user_similarity):
    mean_user_rating = user_item_matrix.mean(axis=1).values
    ratings_diff = (user_item_matrix.values - mean_user_rating[:, np.newaxis])
    pred = mean_user_rating[:, np.newaxis] + user_similarity.dot(ratings_diff) / np.array([np.abs(user_similarity).sum(axis=1)]).T
    return pd.DataFrame(pred, index=user_item_matrix.index, columns=user_item_matrix.columns)

# Dự đoán rating
predicted_ratings = predict_ratings(user_item_matrix, user_similarity)

# Hàm lấy top-N khuyến nghị cho mỗi user
def get_top_n_recommendations(predicted_ratings, n=10):
    top_n = {}
    for user in predicted_ratings.index:
        user_ratings = predicted_ratings.loc[user].sort_values(ascending=False)
        top_n[user] = user_ratings.head(n).index.tolist()
    return top_n

# Lấy top-10 khuyến nghị
top_n_recommendations = get_top_n_recommendations(predicted_ratings, n=10)

KeyError: 'item_id'